# Parameter Golf — Training Notebook

**Instructions:**
1. Runtime → Change runtime type → **A100 GPU**
2. Run all cells (Ctrl+F9)
3. Results will be printed at the end and saved to `experiments.jsonl`

Training takes ~10 minutes. Sliding window eval takes another ~5 minutes.

In [ ]:
# ============================================================
# CONFIG — Edit this cell to change experiment settings
# ============================================================
FORK_REPO = "jxdai2007/parameter-golf"
BRANCH = "main"
RUN_ID = "exp005_11L_tuned"

# Override any env vars here (leave empty dict for defaults from train_gpt.py)
ENV_OVERRIDES = {
    # "NUM_LAYERS": "11",
    # "MLP_MULT": "3",
    # "EVAL_STRIDE": "64",
}

In [ ]:
# ============================================================
# SETUP — Clone repo, install deps, download data
# ============================================================
import os, subprocess

# Check GPU
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"GPU: {gpu_info[0]}")

WORK_DIR = "/content/parameter-golf"

if not os.path.exists(WORK_DIR):
    !git clone -b {BRANCH} https://github.com/{FORK_REPO}.git {WORK_DIR}
else:
    !cd {WORK_DIR} && git fetch origin && git reset --hard origin/{BRANCH}
    print("Repo updated to latest.")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

# Install deps
!pip install -q zstandard sentencepiece 2>&1 | tail -1
print("Dependencies installed.")

In [ ]:
# ============================================================
# DOWNLOAD DATA (only runs once, ~2 min)
# ============================================================
DATA_DIR = "./data/datasets/fineweb10B_sp1024"

if not os.path.exists(DATA_DIR):
    !python3 data/cached_challenge_fineweb.py --variant sp1024
    print("Data downloaded.")
else:
    n_shards = len([f for f in os.listdir(DATA_DIR) if 'train' in f])
    print(f"Data already exists ({n_shards} training shards). Skipping download.")

In [ ]:
# ============================================================
# TRAIN — runs for up to 10 minutes
# ============================================================
import time

# Build environment
env = os.environ.copy()
env["RUN_ID"] = RUN_ID
env["DATA_PATH"] = "./data/datasets/fineweb10B_sp1024/"
env["TOKENIZER_PATH"] = "./data/tokenizers/fineweb_1024_bpe.model"
env["VOCAB_SIZE"] = "1024"
env.update(ENV_OVERRIDES)

cmd = ["torchrun", "--standalone", "--nproc_per_node=1", "train_gpt.py"]

print(f"Starting training: {RUN_ID}")
print(f"Overrides: {ENV_OVERRIDES if ENV_OVERRIDES else 'none (using defaults)'}")
print("=" * 60)

t0 = time.time()
proc = subprocess.Popen(
    cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True
)

output_lines = []
for line in proc.stdout:
    line = line.rstrip()
    output_lines.append(line)
    # Print key lines live
    if any(k in line for k in ["step:", "val_bpb", "stopping", "peak memory",
                                "Serialized", "Total submission", "final_",
                                "sliding_window", "model_params", "Error", "failed"]):
        print(line)

proc.wait()
elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"Done in {elapsed:.0f}s ({elapsed/60:.1f} min). Exit code: {proc.returncode}")

In [ ]:
# ============================================================
# RESULTS SUMMARY
# ============================================================
import re, json

full_output = "\n".join(output_lines)

# Extract key metrics
def extract(pattern, text, default="N/A"):
    m = re.search(pattern, text)
    return m.group(1) if m else default

params = extract(r"model_params:(\d+)", full_output)
roundtrip_bpb = extract(r"final_int6.*?roundtrip_exact.*?val_bpb:([\d.]+)", full_output)
sliding_bpb = extract(r"sliding_window_exact.*?val_bpb:([\d.]+)", full_output)
artifact_size = extract(r"Total submission size.*?:\s*(\d+)", full_output)
steps = extract(r"stopping_early.*?step:(\d+)", full_output)

print("\n" + "=" * 60)
print(f"  Run ID:          {RUN_ID}")
print(f"  Parameters:      {params}")
print(f"  Steps completed: {steps}")
print(f"  Artifact size:   {artifact_size} bytes")
print(f"  Roundtrip BPB:   {roundtrip_bpb}")
print(f"  Sliding BPB:     {sliding_bpb}")
print("=" * 60)

# Log to experiments.jsonl
exp = {
    "id": RUN_ID,
    "description": RUN_ID,
    "bpb": float(roundtrip_bpb) if roundtrip_bpb != "N/A" else None,
    "bpb_sliding": float(sliding_bpb) if sliding_bpb != "N/A" else None,
    "artifact_bytes": int(artifact_size) if artifact_size != "N/A" else None,
    "params": int(params) if params != "N/A" else None,
    "steps": int(steps) if steps != "N/A" else None,
    "gpus": gpu_info[0].strip(),
    "overrides": ENV_OVERRIDES,
}
with open("experiments.jsonl", "a") as f:
    f.write(json.dumps(exp) + "\n")
print(f"\nLogged to experiments.jsonl")

In [ ]:
# ============================================================
# VIEW ALL EXPERIMENTS
# ============================================================
if os.path.exists("experiments.jsonl"):
    print(f"{'ID':<30} {'BPB':>8} {'Sliding':>8} {'Artifact':>12} {'Steps':>6}")
    print("-" * 70)
    for line in open("experiments.jsonl"):
        e = json.loads(line)
        print(f"{e.get('id','?'):<30} {e.get('bpb','?'):>8} {e.get('bpb_sliding','?'):>8} {e.get('artifact_bytes','?'):>12} {e.get('steps','?'):>6}")